# Segunda hipótesis: Nivel 1 entrenado con DU binarizado

Este notebook desarrolla una segunda hipótesis para el sistema en cascada.

En el planteamiento inicial, el Nivel 1 se entrenó únicamente con DATD para distinguir entre publicaciones sanas y enfermas. Al proceder este conjunto de Twitter, el modelo aprendía esta clasificación a partir de mensajes cortos. Sin embargo, el Dataset Unificado utilizado para evaluar la cascada también contiene publicaciones más largas procedentes de Reddit.

Para intentar mejorar la capacidad de generalización del primer nivel, en esta segunda hipótesis se utiliza el Dataset Unificado (DU) binarizado. De esta forma, el modelo dispone durante el entrenamiento de publicaciones sanas y enfermas procedentes de Twitter, así como de publicaciones sanas y enfermas procedentes de Reddit.

Las etiquetas del DU se transforman de la siguiente forma:

- 0 = sano
- 1 = enfermo

Las clases originales 1 = depresión y 2 = suicidio se fusionan en una única clase 1 = enfermo.

El Nivel 2 no se modifica y se reutilizan los modelos entrenados anteriormente con SDCNL para distinguir entre depresión y suicidio.

Esta segunda hipótesis modifica únicamente el sistema en cascada. Los modelos de clasificación directa entrenados anteriormente con el DU original de tres clases no se vuelven a entrenar, ya que su planteamiento no ha cambiado. Sus resultados se mantienen como referencia para comparar el rendimiento de la cascada original con el obtenido tras modificar su Nivel 1.

In [ ]:
# =====================================================
# 1. IMPORTACIONES
# =====================================================

import os
import gc
import random

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import optuna

from sklearn.metrics import (
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed
)

print("Librerías importadas correctamente.")

In [ ]:
# =====================================================
# 2. REPRODUCIBILIDAD Y DISPOSITIVO
# =====================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Semilla:", SEED)
print("Dispositivo:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# =====================================================
# 3. RUTAS
# =====================================================

RAIZ_ORIGINAL = r"."

# Datos ya existentes
RUTA_DATOS_ORIGINAL = os.path.join(
    RAIZ_ORIGINAL,
    "Bases de Datos Splits"
)

# Modelos ya existentes de la primera hipótesis
RUTA_MODELOS_ORIGINAL = os.path.join(
    RAIZ_ORIGINAL,
    "Modelos Definitivos"
)

# Resultados ya existentes
RUTA_RESULTADOS_ORIGINAL = os.path.join(
    RAIZ_ORIGINAL,
    "Resultados Definitivos"
)


# =====================================================
# SEGUNDA HIPÓTESIS
# =====================================================

RAIZ_H2 = r"./Segunda Hipótesis"

RUTA_DATOS_H2 = os.path.join(
    RAIZ_H2,
    "Bases de Datos Splits"
)

RUTA_OPTUNA_H2 = os.path.join(
    RAIZ_H2,
    "Resultados Optuna"
)

RUTA_CHECKPOINTS_H2 = os.path.join(
    RAIZ_H2,
    "Checkpoints"
)

RUTA_MODELOS_H2 = os.path.join(
    RAIZ_H2,
    "Modelos Definitivos"
)

RUTA_RESULTADOS_H2 = os.path.join(
    RAIZ_H2,
    "Resultados Definitivos"
)


for ruta in [
    RUTA_DATOS_H2,
    RUTA_OPTUNA_H2,
    RUTA_CHECKPOINTS_H2,
    RUTA_MODELOS_H2,
    RUTA_RESULTADOS_H2
]:
    os.makedirs(ruta, exist_ok=True)

print("Carpetas creadas correctamente.")
print(RAIZ_H2)

## Carga del Dataset Unificado

Para mantener la comparabilidad con el experimento anterior, se reutilizan los mismos conjuntos de entrenamiento, validación y prueba de DU.

No se vuelve a realizar una nueva división.

Se utilizarán:

- `train_DU_balanceado.csv`
- `val_DU.csv`
- `test_DU.csv`

In [ ]:
# =====================================================
# 4. CARGAR SPLITS DE DU
# =====================================================

ruta_train_DU = os.path.join(
    RUTA_DATOS_ORIGINAL,
    "train_DU_balanceado.csv"
)

ruta_val_DU = os.path.join(
    RUTA_DATOS_ORIGINAL,
    "val_DU.csv"
)

ruta_test_DU = os.path.join(
    RUTA_DATOS_ORIGINAL,
    "test_DU.csv"
)

train_DU = pd.read_csv(ruta_train_DU)
val_DU = pd.read_csv(ruta_val_DU)
test_DU = pd.read_csv(ruta_test_DU)

print("Train:", train_DU.shape)
print("Validation:", val_DU.shape)
print("Test:", test_DU.shape)

print("\nTRAIN:")
print(train_DU["Label"].value_counts().sort_index())

print("\nVALIDATION:")
print(val_DU["Label"].value_counts().sort_index())

print("\nTEST:")
print(test_DU["Label"].value_counts().sort_index())

In [ ]:
# =====================================================
# 5. BINARIZAR DU
# =====================================================

def binarizar_DU(df):

    df_bin = df.copy()

    # Guardamos la etiqueta original
    df_bin["Label_original_DU"] = df_bin["Label"]

    # 0 = sano
    # 1 y 2 = enfermo
    df_bin["Label"] = df_bin["Label"].map({
        0: 0,
        1: 1,
        2: 1
    })

    if df_bin["Label"].isna().any():
        raise ValueError("Se encontraron etiquetas no reconocidas.")

    df_bin["Label"] = df_bin["Label"].astype(int)

    return df_bin


train_DU_bin = binarizar_DU(train_DU)
val_DU_bin = binarizar_DU(val_DU)
test_DU_bin = binarizar_DU(test_DU)

print("DU binarizado correctamente.")

print("\nTRAIN BINARIO:")
print(train_DU_bin["Label"].value_counts().sort_index())

print("\nVALIDATION BINARIO:")
print(val_DU_bin["Label"].value_counts().sort_index())

print("\nTEST BINARIO:")
print(test_DU_bin["Label"].value_counts().sort_index())

In [ ]:
# =====================================================
# 6. COMPROBAR SOURCE POR CLASE
# =====================================================

for nombre, df in [
    ("TRAIN", train_DU_bin),
    ("VALIDATION", val_DU_bin),
    ("TEST", test_DU_bin)
]:

    print("\n==============================")
    print(nombre)
    print("==============================")

    tabla = pd.crosstab(
        df["Source"],
        df["Label"],
        margins=True
    )

    tabla = tabla.rename(
        columns={
            0: "Sano",
            1: "Enfermo"
        }
    )

    display(tabla)

In [ ]:
# =====================================================
# 7. GUARDAR SPLITS BINARIOS
# =====================================================

train_DU_bin.to_csv(
    os.path.join(
        RUTA_DATOS_H2,
        "train_DU_binario_balanceado.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

val_DU_bin.to_csv(
    os.path.join(
        RUTA_DATOS_H2,
        "val_DU_binario.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

test_DU_bin.to_csv(
    os.path.join(
        RUTA_DATOS_H2,
        "test_DU_binario.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

print("Splits binarios guardados.")

## Tokenización

Se mantiene la misma configuración utilizada en los experimentos anteriores:

- longitud máxima: 128 tokens
- padding hasta 128
- truncamiento de textos superiores a 128 tokens

Cada Transformer utilizará su propio tokenizador.

In [ ]:
# =====================================================
# 8. DATASET Y MÉTRICA
# =====================================================

MAX_LENGTH = 128


class DatasetTexto(Dataset):

    def __init__(self, tokens, labels):
        self.tokens = tokens
        self.labels = labels.tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        return {
            "input_ids": torch.tensor(
                self.tokens["input_ids"][idx]
            ),

            "attention_mask": torch.tensor(
                self.tokens["attention_mask"][idx]
            ),

            "labels": torch.tensor(
                self.labels[idx],
                dtype=torch.long
            )
        }


def compute_metrics_binario(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    f1 = f1_score(
        labels,
        predictions,
        average="binary",
        pos_label=1
    )

    return {
        "f1": f1
    }

In [ ]:
# =====================================================
# 9. PREPARAR DATASETS PARA CADA TRANSFORMER
# =====================================================

def preparar_datasets_transformer(nombre_modelo):

    tokenizer = AutoTokenizer.from_pretrained(
        nombre_modelo
    )

    def tokenizar(df):

        return tokenizer(
            df["Text"].tolist(),
            padding="max_length",
            truncation=True,
            max_length=MAX_LENGTH
        )

    tokens_train = tokenizar(train_DU_bin)
    tokens_val = tokenizar(val_DU_bin)
    tokens_test = tokenizar(test_DU_bin)

    dataset_train = DatasetTexto(
        tokens_train,
        train_DU_bin["Label"]
    )

    dataset_val = DatasetTexto(
        tokens_val,
        val_DU_bin["Label"]
    )

    dataset_test = DatasetTexto(
        tokens_test,
        test_DU_bin["Label"]
    )

    return (
        tokenizer,
        dataset_train,
        dataset_val,
        dataset_test
    )

## Ajuste de hiperparámetros

Dado que el conjunto utilizado para entrenar el Nivel 1 ha cambiado, se realiza una nueva búsqueda de hiperparámetros mediante Optuna.

Se mantienen las mismas condiciones utilizadas anteriormente:

- batch size: 8, 16 o 32
- learning rate: 1e-5, 3e-5 o 5e-5
- weight decay: entre 0.01 y 0.1
- épocas: entre 10 y 15
- early stopping: 3 épocas
- warmup steps: 100
- max grad norm: 1.0
- 20 trials

In [ ]:
# =====================================================
# 10. FUNCIÓN OPTUNA
# =====================================================

def ejecutar_optuna_nivel1(
    nombre_modelo,
    nombre_corto,
    dataset_train,
    dataset_val
):

    def objective(trial):

        batch_size = trial.suggest_categorical(
            "batch_size",
            [8, 16, 32]
        )

        learning_rate = trial.suggest_categorical(
            "learning_rate",
            [1e-5, 3e-5, 5e-5]
        )

        weight_decay = trial.suggest_float(
            "weight_decay",
            0.01,
            0.1
        )

        epochs = trial.suggest_int(
            "epochs",
            10,
            15
        )

        set_seed(SEED)

        model = AutoModelForSequenceClassification.from_pretrained(
            nombre_modelo,
            num_labels=2,
            dtype=torch.float32
        )

        ruta_trial = os.path.join(
            RUTA_CHECKPOINTS_H2,
            "Optuna",
            nombre_corto,
            f"trial_{trial.number}"
        )

        args = TrainingArguments(

            output_dir=ruta_trial,

            eval_strategy="epoch",
            save_strategy="epoch",

            load_best_model_at_end=True,

            metric_for_best_model="f1",
            greater_is_better=True,

            learning_rate=learning_rate,

            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=16,

            weight_decay=weight_decay,

            num_train_epochs=epochs,

            warmup_steps=100,

            max_grad_norm=1.0,

            seed=SEED,

            save_total_limit=1,

            logging_strategy="epoch",

            report_to="none"
        )

        trainer = Trainer(

            model=model,

            args=args,

            train_dataset=dataset_train,

            eval_dataset=dataset_val,

            compute_metrics=compute_metrics_binario,

            callbacks=[
                EarlyStoppingCallback(
                    early_stopping_patience=3
                )
            ]
        )

        trainer.train()

        mejor_f1 = trainer.state.best_metric

        del trainer
        del model

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return mejor_f1


    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(
            seed=SEED
        )
    )

    study.optimize(
        objective,
        n_trials=20
    )

    resultados = study.trials_dataframe()

    resultados.to_csv(
        os.path.join(
            RUTA_OPTUNA_H2,
            f"resultados_optuna_{nombre_corto}_DU_binario.csv"
        ),
        index=False,
        encoding="utf-8-sig"
    )

    print("Mejor F1:", study.best_value)
    print("Mejores hiperparámetros:")
    print(study.best_params)

    return study

In [ ]:
# =====================================================
# 11. PREPARAR ROBERTA
# =====================================================

(
    tokenizer_roberta,
    dataset_train_roberta,
    dataset_val_roberta,
    dataset_test_roberta
) = preparar_datasets_transformer(
    "roberta-base"
)

print("RoBERTa preparado.")

In [ ]:
# =====================================================
# 12. OPTUNA ROBERTA
# =====================================================

study_roberta = ejecutar_optuna_nivel1(

    nombre_modelo="roberta-base",

    nombre_corto="RoBERTa",

    dataset_train=dataset_train_roberta,

    dataset_val=dataset_val_roberta
)

In [ ]:
# =====================================================
# GUARDAR MEJOR CONFIGURACIÓN - ROBERTA
# =====================================================

mejor_roberta = pd.DataFrame([{
    "Transformer": "RoBERTa",
    "Dataset": "DU binarizado",
    "F1_Validation": study_roberta.best_value,
    "batch_size": study_roberta.best_params["batch_size"],
    "learning_rate": study_roberta.best_params["learning_rate"],
    "weight_decay": study_roberta.best_params["weight_decay"],
    "epochs": study_roberta.best_params["epochs"]
}])

display(mejor_roberta)

ruta_mejor_roberta = os.path.join(
    RUTA_OPTUNA_H2,
    "mejor_configuracion_RoBERTa_DU_binario.csv"
)

mejor_roberta.to_csv(
    ruta_mejor_roberta,
    index=False,
    encoding="utf-8-sig"
)

print("Mejor configuración de RoBERTa guardada en:")
print(ruta_mejor_roberta)

In [ ]:
# =====================================================
# 13. PREPARAR DEBERTA
# =====================================================

(
    tokenizer_deberta,
    dataset_train_deberta,
    dataset_val_deberta,
    dataset_test_deberta
) = preparar_datasets_transformer(
    "microsoft/deberta-v3-base"
)

print("DeBERTa preparado.")

In [ ]:
# =====================================================
# 14. OPTUNA DEBERTA
# =====================================================

study_deberta = ejecutar_optuna_nivel1(

    nombre_modelo="microsoft/deberta-v3-base",

    nombre_corto="DeBERTa",

    dataset_train=dataset_train_deberta,

    dataset_val=dataset_val_deberta
)

In [ ]:
# =====================================================
# GUARDAR MEJOR CONFIGURACIÓN - DEBERTA
# =====================================================

mejor_deberta = pd.DataFrame([{
    "Transformer": "DeBERTa",
    "Dataset": "DU binarizado",
    "F1_Validation": study_deberta.best_value,
    "batch_size": study_deberta.best_params["batch_size"],
    "learning_rate": study_deberta.best_params["learning_rate"],
    "weight_decay": study_deberta.best_params["weight_decay"],
    "epochs": study_deberta.best_params["epochs"]
}])

display(mejor_deberta)

ruta_mejor_deberta = os.path.join(
    RUTA_OPTUNA_H2,
    "mejor_configuracion_DeBERTa_DU_binario.csv"
)

mejor_deberta.to_csv(
    ruta_mejor_deberta,
    index=False,
    encoding="utf-8-sig"
)

print("Mejor configuración de DeBERTa guardada en:")
print(ruta_mejor_deberta)

## Entrenamiento con mejores Hiperparámetros del nuevo Nivel 1

Una vez finalizada la búsqueda de hiperparámetros con Optuna, se selecciona la mejor configuración obtenida para cada Transformer.

A continuación, RoBERTa Base y DeBERTa v3 Base se cargan nuevamente desde sus modelos preentrenados originales y se realiza un nuevo fine-tuning utilizando el DU binarizado. Cada Transformer se entrena con su propia configuración de hiperparámetros seleccionada por Optuna.

El objetivo de este entrenamiento es obtener los dos clasificadores que se utilizarán como Nivel 1 de las nuevas cascadas:

- RoBERTa entrenado con DU binarizado para clasificar sano/enfermo.
- DeBERTa entrenado con DU binarizado para clasificar sano/enfermo.

Durante el entrenamiento, `train_DU_bin` se utiliza para actualizar los pesos y `val_DU_bin` para seleccionar el mejor estado del modelo mediante F1. Una vez seleccionado, se evalúa sobre `test_DU_bin`.

Estos modelos sustituyen únicamente al Nivel 1 de la cascada original. Los modelos del Nivel 2, entrenados con SDCNL para distinguir entre depresión y suicidio, se mantienen sin cambios.

In [ ]:
# =================================================================================
# 15. CARGAR LAS CONFIGURACIONES SELECCIONADAS POR OPTUNA PARA EL NUEVO NIVEL 1
# =================================================================================

ruta_mejor_roberta = os.path.join(
    RUTA_OPTUNA_H2,
    "mejor_configuracion_RoBERTa_DU_binario.csv"
)

ruta_mejor_deberta = os.path.join(
    RUTA_OPTUNA_H2,
    "mejor_configuracion_DeBERTa_DU_binario.csv"
)

mejor_roberta = pd.read_csv(ruta_mejor_roberta)
mejor_deberta = pd.read_csv(ruta_mejor_deberta)

print("Mejor configuración RoBERTa:")
display(mejor_roberta)

print("\nMejor configuración DeBERTa:")
display(mejor_deberta)

In [ ]:
# =====================================================
# 16. PREPARAR HIPERPARÁMETROS GANADORES
# =====================================================

hp_roberta = {
    "batch_size": int(mejor_roberta.loc[0, "batch_size"]),
    "learning_rate": float(mejor_roberta.loc[0, "learning_rate"]),
    "weight_decay": float(mejor_roberta.loc[0, "weight_decay"]),
    "epochs": int(mejor_roberta.loc[0, "epochs"])
}

hp_deberta = {
    "batch_size": int(mejor_deberta.loc[0, "batch_size"]),
    "learning_rate": float(mejor_deberta.loc[0, "learning_rate"]),
    "weight_decay": float(mejor_deberta.loc[0, "weight_decay"]),
    "epochs": int(mejor_deberta.loc[0, "epochs"])
}

print("RoBERTa:")
print(hp_roberta)

print("\nDeBERTa:")
print(hp_deberta)

In [ ]:
# =====================================================
# 17. FUNCIÓN DE ENTRENAMIENTO DEFINITIVO DEL NUEVO NIVEL 1
# =====================================================

def entrenar_nivel1_definitivo(
    nombre_modelo,
    nombre_corto,
    tokenizer,
    dataset_train,
    dataset_val,
    dataset_test,
    hiperparametros
):

    set_seed(SEED)

    print("\n============================================")
    print(f"ENTRENAMIENTO DEFINITIVO - {nombre_corto}")
    print("Nuevo Nivel 1 - DU binarizado")
    print("============================================")

    print("Hiperparámetros:")
    print(hiperparametros)

    # Cargar nuevamente el modelo preentrenado original
    model = AutoModelForSequenceClassification.from_pretrained(
        nombre_modelo,
        num_labels=2,
        dtype=torch.float32
    )

    ruta_checkpoints = os.path.join(
        RUTA_CHECKPOINTS_H2,
        nombre_corto,
        "DU_binario_N1"
    )

    args = TrainingArguments(
        output_dir=ruta_checkpoints,

        eval_strategy="epoch",
        save_strategy="epoch",

        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,

        learning_rate=hiperparametros["learning_rate"],
        per_device_train_batch_size=hiperparametros["batch_size"],
        per_device_eval_batch_size=16,
        weight_decay=hiperparametros["weight_decay"],
        num_train_epochs=hiperparametros["epochs"],

        warmup_steps=100,
        max_grad_norm=1.0,

        seed=SEED,

        save_total_limit=1,
        logging_strategy="epoch",

        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,

        train_dataset=dataset_train,
        eval_dataset=dataset_val,

        compute_metrics=compute_metrics_binario,

        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=3
            )
        ]
    )

    # Entrenamiento
    trainer.train()

    mejor_f1_validacion = trainer.state.best_metric

    # Evaluación final sobre test
    resultados_test = trainer.evaluate(dataset_test)

    f1_test = resultados_test["eval_f1"]

    # Guardar modelo definitivo
    ruta_modelo = os.path.join(
        RUTA_MODELOS_H2,
        f"{nombre_corto}_DU_binario_N1"
    )

    trainer.save_model(ruta_modelo)
    tokenizer.save_pretrained(ruta_modelo)

    print("\n============================================")
    print("RESULTADOS")
    print("============================================")
    print("Mejor F1 validación:", mejor_f1_validacion)
    print("F1 test_DU binario:", f1_test)
    print("Modelo guardado en:")
    print(ruta_modelo)

    # Liberar memoria
    del trainer
    del model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "Transformer": nombre_corto,
        "F1_Validation": mejor_f1_validacion,
        "F1_Test_DU_Binario": f1_test,
        "Ruta_Modelo": ruta_modelo
    }

In [ ]:
# =====================================================
# 18. ENTRENAR ROBERTA COMO NUEVO NIVEL 1 con DU BINARIZADO
# =====================================================

resultado_roberta_n1 = entrenar_nivel1_definitivo(
    nombre_modelo="roberta-base",
    nombre_corto="RoBERTa",

    tokenizer=tokenizer_roberta,

    dataset_train=dataset_train_roberta,
    dataset_val=dataset_val_roberta,
    dataset_test=dataset_test_roberta,

    hiperparametros=hp_roberta
)

resultado_roberta_n1

In [ ]:
# =====================================================
# 19. ENTRENAR DeBERTa COMO NUEVO NIVEL 1 con DU BINARIZADO
# =====================================================

resultado_deberta_n1 = entrenar_nivel1_definitivo(
    nombre_modelo="microsoft/deberta-v3-base",
    nombre_corto="DeBERTa",

    tokenizer=tokenizer_deberta,

    dataset_train=dataset_train_deberta,
    dataset_val=dataset_val_deberta,
    dataset_test=dataset_test_deberta,

    hiperparametros=hp_deberta
)

resultado_deberta_n1

In [ ]:
# =====================================================
# 20. GUARDAR RESULTADOS DE LOS NUEVOS NIVELES 1
# =====================================================

resultados_nuevo_n1 = pd.DataFrame([
    resultado_roberta_n1,
    resultado_deberta_n1
])

display(resultados_nuevo_n1)

ruta_resultados_n1 = os.path.join(
    RUTA_RESULTADOS_H2,
    "resultados_nuevo_nivel1_DU_binario.csv"
)

resultados_nuevo_n1.to_csv(
    ruta_resultados_n1,
    index=False,
    encoding="utf-8-sig"
)

print("Resultados guardados en:")
print(ruta_resultados_n1)

## Construcción de las nuevas cascadas

El segundo nivel de la cascada no se modifica, ya que el análisis realizado sobre la primera hipótesis mostró que su capacidad para distinguir entre depresión y suicidio era adecuada.

Por tanto, las nuevas cascadas combinan:

- Nuevo Nivel 1 entrenado con DU binarizado.
- Nivel 2 original entrenado con SDCNL.

Se vuelven a construir las cuatro combinaciones posibles entre RoBERTa y DeBERTa.

In [ ]:
# =====================================================
# 21. RUTAS DE LOS MODELOS PARA LA NUEVA CASCADA
# =====================================================

# -------------------------------
# NUEVO NIVEL 1
# -------------------------------

RUTA_N1_ROBERTA_NUEVO = os.path.join(
    RUTA_MODELOS_H2,
    "RoBERTa_DU_binario_N1"
)

RUTA_N1_DEBERTA_NUEVO = os.path.join(
    RUTA_MODELOS_H2,
    "DeBERTa_DU_binario_N1"
)


# -------------------------------
# NIVEL 2 ORIGINAL
# -------------------------------

RUTA_N2_ROBERTA = os.path.join(
    RUTA_MODELOS_ORIGINAL,
    "RoBERTa_SDCNL"
)

RUTA_N2_DEBERTA = os.path.join(
    RUTA_MODELOS_ORIGINAL,
    "DeBERTa_SDCNL"
)


print("Nuevo Nivel 1 RoBERTa:")
print(RUTA_N1_ROBERTA_NUEVO)

print("\nNuevo Nivel 1 DeBERTa:")
print(RUTA_N1_DEBERTA_NUEVO)

print("\nNivel 2 RoBERTa:")
print(RUTA_N2_ROBERTA)

print("\nNivel 2 DeBERTa:")
print(RUTA_N2_DEBERTA)

In [ ]:
# =====================================================
# 22. COMPROBAR RUTAS DE MODELOS
# =====================================================

rutas_a_comprobar = {
    "Nuevo N1 RoBERTa": RUTA_N1_ROBERTA_NUEVO,
    "Nuevo N1 DeBERTa": RUTA_N1_DEBERTA_NUEVO,
    "N2 RoBERTa original": RUTA_N2_ROBERTA,
    "N2 DeBERTa original": RUTA_N2_DEBERTA
}

for nombre, ruta in rutas_a_comprobar.items():

    existe = os.path.exists(ruta)

    print(
        f"{nombre}:",
        "OK" if existe else "NO ENCONTRADO"
    )

    print("   ", ruta)

In [ ]:
# =====================================================
# 23. CARGAR MODELOS ENTRENADOS
# =====================================================

def cargar_modelo_entrenado(ruta_modelo):

    tokenizer = AutoTokenizer.from_pretrained(
        ruta_modelo
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        ruta_modelo,
        dtype=torch.float32
    )

    model.to(device)
    model.eval()

    return model, tokenizer

In [ ]:
# =====================================================
# 24. FUNCIÓN DE PREDICCIÓN
# =====================================================

def predecir_textos(
    modelo,
    tokenizer,
    textos,
    batch_size=32
):

    predicciones = []

    for inicio in range(
        0,
        len(textos),
        batch_size
    ):

        lote = textos[
            inicio:inicio + batch_size
        ]

        inputs = tokenizer(
            lote,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        inputs = {
            clave: valor.to(device)
            for clave, valor in inputs.items()
        }

        with torch.no_grad():

            outputs = modelo(**inputs)

        pred_lote = torch.argmax(
            outputs.logits,
            dim=-1
        )

        predicciones.extend(
            pred_lote.cpu().numpy().tolist()
        )

    return predicciones

In [ ]:
# =====================================================
# 25. FUNCIÓN DE CASCADA
# =====================================================

def predecir_cascada(
    textos,
    modelo_n1,
    tokenizer_n1,
    modelo_n2,
    tokenizer_n2
):

    # -----------------------------------------
    # NIVEL 1
    # 0 = sano
    # 1 = enfermo
    # -----------------------------------------

    pred_n1 = predecir_textos(
        modelo_n1,
        tokenizer_n1,
        textos
    )

    pred_finales = np.zeros(
        len(textos),
        dtype=int
    )

    # Textos clasificados como enfermos
    indices_enfermos = [
        i
        for i, pred in enumerate(pred_n1)
        if pred == 1
    ]

    print("Total de textos:", len(textos))
    print(
        "Clasificados como sanos en N1:",
        pred_n1.count(0)
    )
    print(
        "Enviados al Nivel 2:",
        len(indices_enfermos)
    )


    # -----------------------------------------
    # NIVEL 2
    # Internamente:
    # 0 = depresión
    # 1 = suicidio
    # -----------------------------------------

    if len(indices_enfermos) > 0:

        textos_enfermos = [
            textos[i]
            for i in indices_enfermos
        ]

        pred_n2 = predecir_textos(
            modelo_n2,
            tokenizer_n2,
            textos_enfermos
        )

        # Convertir de etiquetas N2:
        #
        # 0 -> 1 = depresión en DU
        # 1 -> 2 = suicidio en DU

        for indice_original, pred in zip(
            indices_enfermos,
            pred_n2
        ):

            pred_finales[indice_original] = pred + 1

    return pred_finales

In [ ]:
# =====================================================
# 26. PREPARAR TEST_DU MULTICLASE
# =====================================================

test_DU_multiclase = pd.read_csv(
    os.path.join(
        RUTA_DATOS_ORIGINAL,
        "test_DU.csv"
    )
)

textos_test_DU = (
    test_DU_multiclase["Text"]
    .astype(str)
    .tolist()
)

y_test_DU = (
    test_DU_multiclase["Label"]
    .astype(int)
    .to_numpy()
)

print("Número de textos:", len(test_DU_multiclase))

print("\nDistribución de clases:")
print(
    test_DU_multiclase["Label"]
    .value_counts()
    .sort_index()
)

In [ ]:
# =====================================================
# 27. EVALUAR UNA NUEVA CASCADA
# =====================================================

def evaluar_cascada(
    nombre_n1,
    ruta_n1,
    nombre_n2,
    ruta_n2
):

    print("\n============================================")
    print(f"CASCADA: {nombre_n1} -> {nombre_n2}")
    print("============================================")

    modelo_n1, tokenizer_n1 = cargar_modelo_entrenado(
        ruta_n1
    )

    modelo_n2, tokenizer_n2 = cargar_modelo_entrenado(
        ruta_n2
    )

    predicciones = predecir_cascada(
        textos_test_DU,

        modelo_n1,
        tokenizer_n1,

        modelo_n2,
        tokenizer_n2
    )

    macro_f1 = f1_score(
        y_test_DU,
        predicciones,
        average="macro"
    )

    print("\nMacro-F1 test_DU:")
    print(macro_f1)

    del modelo_n1
    del tokenizer_n1
    del modelo_n2
    del tokenizer_n2

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return macro_f1, predicciones

In [ ]:
# =====================================================
# 28. NUEVA CASCADA 1 ROBERTA -> ROBERTA
# =====================================================

f1_RR, pred_RR = evaluar_cascada(
    nombre_n1="RoBERTa",
    ruta_n1=RUTA_N1_ROBERTA_NUEVO,

    nombre_n2="RoBERTa",
    ruta_n2=RUTA_N2_ROBERTA
)

In [ ]:
# =====================================================
# 29. NUEVA CASCADA 2 ROBERTA -> DEBERTA
# =====================================================

f1_RD, pred_RD = evaluar_cascada(
    nombre_n1="RoBERTa",
    ruta_n1=RUTA_N1_ROBERTA_NUEVO,

    nombre_n2="DeBERTa",
    ruta_n2=RUTA_N2_DEBERTA
)

In [ ]:
# =====================================================
# 30. NUEVA CASCADA 3 DEBERTA -> ROBERTA
# =====================================================

f1_DR, pred_DR = evaluar_cascada(
    nombre_n1="DeBERTa",
    ruta_n1=RUTA_N1_DEBERTA_NUEVO,

    nombre_n2="RoBERTa",
    ruta_n2=RUTA_N2_ROBERTA
)

In [ ]:
# =====================================================
# 31. NUEVA CASCADA 4 DEBERTA -> DEBERTA
# =====================================================

f1_DD, pred_DD = evaluar_cascada(
    nombre_n1="DeBERTa",
    ruta_n1=RUTA_N1_DEBERTA_NUEVO,

    nombre_n2="DeBERTa",
    ruta_n2=RUTA_N2_DEBERTA
)

In [ ]:
# =====================================================
# 32. RESULTADOS DE LAS CUATRO NUEVAS CASCADAS
# =====================================================

resultados_cascadas_h2 = pd.DataFrame({
    "Nivel_1": [
        "RoBERTa",
        "RoBERTa",
        "DeBERTa",
        "DeBERTa"
    ],

    "Entrenamiento_Nivel_1": [
        "DU binarizado",
        "DU binarizado",
        "DU binarizado",
        "DU binarizado"
    ],

    "Nivel_2": [
        "RoBERTa",
        "DeBERTa",
        "RoBERTa",
        "DeBERTa"
    ],

    "Macro_F1_Test_DU": [
        f1_RR,
        f1_RD,
        f1_DR,
        f1_DD
    ]
})

resultados_cascadas_h2 = resultados_cascadas_h2.sort_values(
    "Macro_F1_Test_DU",
    ascending=False
)

display(resultados_cascadas_h2)

ruta_resultados_cascadas = os.path.join(
    RUTA_RESULTADOS_H2,
    "resultados_nuevas_cascadas.csv"
)

resultados_cascadas_h2.to_csv(
    ruta_resultados_cascadas,
    index=False,
    encoding="utf-8-sig"
)

print("Guardado en:")
print(ruta_resultados_cascadas)

In [ ]:
###################################################################################################################################################

# Bloque de diagnóstico y comparación de la segunda hipótesis

Una vez construidas y evaluadas las nuevas cascadas, se analiza si la modificación realizada en el Nivel 1 ha corregido el problema detectado en el planteamiento original.

En este bloque no se realiza ningún nuevo entrenamiento. Se utilizan los modelos ya guardados y los resultados obtenidos anteriormente.

El análisis se divide en varios diagnósticos:

1. Comparación entre el Nivel 1 original, entrenado con DATD, y el nuevo Nivel 1, entrenado con DU binarizado.
2. Análisis de los errores del Nivel 1 según la procedencia de los textos.
3. Análisis específico de los falsos negativos procedentes de SDCNL.
4. Comparación global entre los modelos directos, las cascadas originales y las nuevas cascadas.
5. Análisis detallado de la mejor cascada obtenida en la segunda hipótesis.

El objetivo final es comprobar si el cambio realizado en el Nivel 1 mejora el funcionamiento de la cascada completa y cómo se sitúa esta nueva estrategia frente a la clasificación directa.

In [ ]:
# =====================================================
# 33. PREPARACIÓN DEL BLOQUE DE DIAGNÓSTICO
# =====================================================
#
# Esta celda permite ejecutar el bloque de diagnóstico
# sin volver a ejecutar Optuna ni los entrenamientos.
#
# Se recuperan:
# - rutas de los modelos ya guardados;
# - test_DU original;
# - resultados de las nuevas cascadas;
# - funciones necesarias para cargar modelos y predecir.
#
# NO SE ENTRENA NINGÚN MODELO EN ESTA CELDA.
# =====================================================

import os
import gc
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

from sklearn.metrics import (
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)


# -----------------------------------------------------
# DISPOSITIVO
# -----------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Dispositivo:", device)


# -----------------------------------------------------
# LONGITUD MÁXIMA UTILIZADA EN TODO EL PROYECTO
# -----------------------------------------------------

MAX_LENGTH = 128


# -----------------------------------------------------
# RUTAS DE LA PRIMERA HIPÓTESIS
# -----------------------------------------------------

RUTA_DATOS_ORIGINAL = (
    "./Bases de Datos Splits"
)

RUTA_MODELOS_ORIGINAL = (
    "./Modelos Definitivos"
)

RUTA_RESULTADOS_ORIGINAL = (
    "./Resultados Definitivos"
)


# -----------------------------------------------------
# RUTAS DE LA SEGUNDA HIPÓTESIS
# -----------------------------------------------------

RUTA_H2 = (
    "./Segunda Hipótesis"
)

RUTA_MODELOS_H2 = os.path.join(
    RUTA_H2,
    "Modelos Definitivos"
)

RUTA_RESULTADOS_H2 = os.path.join(
    RUTA_H2,
    "Resultados Definitivos"
)


# -----------------------------------------------------
# MODELOS DEL NIVEL 1 ORIGINAL
# Entrenados con DATD
# -----------------------------------------------------

RUTA_N1_ROBERTA_ORIGINAL = os.path.join(
    RUTA_MODELOS_ORIGINAL,
    "RoBERTa_DATD"
)

RUTA_N1_DEBERTA_ORIGINAL = os.path.join(
    RUTA_MODELOS_ORIGINAL,
    "DeBERTa_DATD"
)


# -----------------------------------------------------
# NUEVOS MODELOS DEL NIVEL 1
# Entrenados con DU binarizado
# -----------------------------------------------------

RUTA_N1_ROBERTA_NUEVO = os.path.join(
    RUTA_MODELOS_H2,
    "RoBERTa_DU_binario_N1"
)

RUTA_N1_DEBERTA_NUEVO = os.path.join(
    RUTA_MODELOS_H2,
    "DeBERTa_DU_binario_N1"
)


# -----------------------------------------------------
# MODELOS DEL NIVEL 2
# Se mantienen los originales entrenados con SDCNL
# -----------------------------------------------------

RUTA_N2_ROBERTA = os.path.join(
    RUTA_MODELOS_ORIGINAL,
    "RoBERTa_SDCNL"
)

RUTA_N2_DEBERTA = os.path.join(
    RUTA_MODELOS_ORIGINAL,
    "DeBERTa_SDCNL"
)


# -----------------------------------------------------
# CARGAR TEST_DU ORIGINAL
# -----------------------------------------------------

test_DU_multiclase = pd.read_csv(
    os.path.join(
        RUTA_DATOS_ORIGINAL,
        "test_DU.csv"
    )
)

textos_test_DU = (
    test_DU_multiclase["Text"]
    .astype(str)
    .tolist()
)

y_test_DU = (
    test_DU_multiclase["Label"]
    .astype(int)
    .to_numpy()
)


# -----------------------------------------------------
# CARGAR RESULTADOS YA OBTENIDOS DE LAS NUEVAS CASCADAS
# -----------------------------------------------------

resultados_cascadas_h2 = pd.read_csv(
    os.path.join(
        RUTA_RESULTADOS_H2,
        "resultados_nuevas_cascadas.csv"
    )
)


# -----------------------------------------------------
# FUNCIÓN PARA CARGAR UN MODELO YA ENTRENADO
# -----------------------------------------------------

def cargar_modelo_entrenado(ruta_modelo):

    tokenizer = AutoTokenizer.from_pretrained(
        ruta_modelo
    )

    modelo = AutoModelForSequenceClassification.from_pretrained(
        ruta_modelo,
        dtype=torch.float32
    )

    modelo.to(device)
    modelo.eval()

    return modelo, tokenizer


# -----------------------------------------------------
# FUNCIÓN PARA REALIZAR PREDICCIONES
# -----------------------------------------------------

def predecir_textos(
    modelo,
    tokenizer,
    textos,
    batch_size=32
):

    predicciones = []

    for inicio in range(
        0,
        len(textos),
        batch_size
    ):

        lote = textos[
            inicio:inicio + batch_size
        ]

        inputs = tokenizer(
            lote,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        inputs = {
            clave: valor.to(device)
            for clave, valor in inputs.items()
        }

        with torch.no_grad():
            outputs = modelo(**inputs)

        pred_lote = torch.argmax(
            outputs.logits,
            dim=-1
        )

        predicciones.extend(
            pred_lote.cpu().numpy().tolist()
        )

    return predicciones


print("\nDatos recuperados correctamente.")
print("Número de textos de test_DU:", len(test_DU_multiclase))

print("\nDistribución original de test_DU:")
print(
    test_DU_multiclase["Label"]
    .value_counts()
    .sort_index()
)

print("\nResultados de las nuevas cascadas:")
display(resultados_cascadas_h2)

## Diagnóstico 1: Rendimiento del Nivel 1 original frente al nuevo Nivel 1

La primera comprobación consiste en determinar si el cambio realizado en el entrenamiento del Nivel 1 ha mejorado realmente su capacidad para distinguir entre publicaciones sanas y enfermas.

Se comparan cuatro modelos:

- RoBERTa entrenado con DATD, correspondiente al Nivel 1 original.
- DeBERTa entrenado con DATD, correspondiente al Nivel 1 original.
- RoBERTa entrenado con DU binarizado, correspondiente al nuevo Nivel 1.
- DeBERTa entrenado con DU binarizado, correspondiente al nuevo Nivel 1.

Los cuatro modelos se evalúan sobre exactamente las mismas publicaciones de `test_DU`.

Como `test_DU` contiene originalmente las clases sano, depresión y suicidio, sus etiquetas deben transformarse al problema binario que resuelve el Nivel 1:

- 0 = sano
- 1 = enfermo

Las clases depresión y suicidio se agrupan dentro de la clase enfermo.

Se presta especial atención a los falsos negativos, ya que corresponden a publicaciones realmente enfermas clasificadas como sanas. En una cascada, estos casos no pasan al Nivel 2 y no pueden recuperarse posteriormente.

In [ ]:
# =====================================================
# 34. PREPARAR TEST_DU PARA EVALUAR EL NIVEL 1
# =====================================================
#
# ¿Qué necesitamos comprobar?
#
# El Nivel 1 únicamente distingue:
#
# 0 = sano
# 1 = enfermo
#
# Sin embargo, test_DU contiene:
#
# 0 = sano
# 1 = depresión
# 2 = suicidio
#
# Por tanto:
#
# 0       -> 0 = sano
# 1 y 2   -> 1 = enfermo
# =====================================================

y_test_DU_binario = np.where(
    y_test_DU == 0,
    0,
    1
)

print("Distribución binaria de test_DU:")

valores, cantidades = np.unique(
    y_test_DU_binario,
    return_counts=True
)

for valor, cantidad in zip(
    valores,
    cantidades
):
    print(valor, ":", cantidad)

In [ ]:
# =====================================================
# 35. FUNCIÓN DE EVALUACIÓN DEL NIVEL 1
# =====================================================
#
# ¿Qué pregunta permite responder esta función?
#
# ¿Qué tal distingue un determinado modelo entre
# publicaciones sanas y enfermas cuando recibe test_DU?
#
# Para cada modelo se calculan:
#
# - F1 de la clase enfermo.
# - Macro-F1.
# - Verdaderos negativos (TN).
# - Falsos positivos (FP).
# - Falsos negativos (FN).
# - Verdaderos positivos (TP).
#
# Los FN son especialmente importantes porque corresponden
# a enfermos clasificados como sanos que no pasarían a N2.
# =====================================================

def evaluar_nivel1_test_DU(
    nombre,
    ruta_modelo
):

    print("\n============================================")
    print(nombre)
    print("============================================")

    # Cargar modelo ya entrenado
    modelo, tokenizer = cargar_modelo_entrenado(
        ruta_modelo
    )

    # Obtener predicciones sano/enfermo sobre test_DU
    predicciones = np.array(
        predecir_textos(
            modelo,
            tokenizer,
            textos_test_DU
        )
    )

    # -------------------------------------------------
    # F1 DE LA CLASE ENFERMO
    # -------------------------------------------------

    f1_enfermo = f1_score(
        y_test_DU_binario,
        predicciones,
        average="binary",
        pos_label=1
    )

    # -------------------------------------------------
    # MACRO-F1 DEL PROBLEMA BINARIO
    # -------------------------------------------------

    macro_f1 = f1_score(
        y_test_DU_binario,
        predicciones,
        average="macro"
    )

    # -------------------------------------------------
    # MATRIZ DE CONFUSIÓN
    # -------------------------------------------------

    cm = confusion_matrix(
        y_test_DU_binario,
        predicciones,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    print("F1 clase enfermo:", f1_enfermo)
    print("Macro-F1:", macro_f1)

    print("\nMatriz de confusión:")
    print(cm)

    print("\nTN:", tn)
    print("FP:", fp)
    print("FN:", fn)
    print("TP:", tp)

    # Liberar el modelo de memoria
    del modelo
    del tokenizer

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Guardar los resultados para utilizarlos
    # posteriormente en los demás diagnósticos
    return {
        "Modelo": nombre,
        "F1_Enfermo": f1_enfermo,
        "Macro_F1": macro_f1,
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "Predicciones": predicciones
    }

In [ ]:
# =====================================================
# 36. NIVEL 1 ORIGINAL - ROBERTA
# =====================================================
#
# ¿Cómo funcionaba el Nivel 1 de RoBERTa cuando estaba
# entrenado únicamente con DATD?
# =====================================================

diag_roberta_original = evaluar_nivel1_test_DU(
    nombre="RoBERTa - Nivel 1 original - DATD",
    ruta_modelo=RUTA_N1_ROBERTA_ORIGINAL
)

In [ ]:
# =====================================================
# 37. NIVEL 1 ORIGINAL - DEBERTA
# =====================================================
#
# ¿Cómo funcionaba el Nivel 1 de DeBERTa cuando estaba
# entrenado únicamente con DATD?
# =====================================================

diag_deberta_original = evaluar_nivel1_test_DU(
    nombre="DeBERTa - Nivel 1 original - DATD",
    ruta_modelo=RUTA_N1_DEBERTA_ORIGINAL
)

In [ ]:
# =====================================================
# 38. NUEVO NIVEL 1 - ROBERTA
# =====================================================
#
# ¿Cómo funciona ahora RoBERTa después de sustituir
# DATD por DU binarizado en el entrenamiento de N1?
# =====================================================

diag_roberta_nuevo = evaluar_nivel1_test_DU(
    nombre="RoBERTa - Nuevo Nivel 1 - DU binarizado",
    ruta_modelo=RUTA_N1_ROBERTA_NUEVO
)

In [ ]:
# =====================================================
# 39. NUEVO NIVEL 1 - DEBERTA
# =====================================================
#
# ¿Cómo funciona ahora DeBERTa después de sustituir
# DATD por DU binarizado en el entrenamiento de N1?
# =====================================================

diag_deberta_nuevo = evaluar_nivel1_test_DU(
    nombre="DeBERTa - Nuevo Nivel 1 - DU binarizado",
    ruta_modelo=RUTA_N1_DEBERTA_NUEVO
)

In [ ]:
# =====================================================
# 40. RESUMEN DEL DIAGNÓSTICO 1
# NIVEL 1 ORIGINAL VS NUEVO NIVEL 1
# =====================================================
#
# ¿El nuevo Nivel 1 funciona mejor que el original?
#
# Se reúnen los resultados de los cuatro modelos para
# comparar directamente:
#
# - F1 de la clase enfermo
# - Macro-F1
# - TN
# - FP
# - FN
# - TP
# =====================================================

diagnosticos_n1 = [
    diag_roberta_original,
    diag_deberta_original,
    diag_roberta_nuevo,
    diag_deberta_nuevo
]

tabla_n1 = pd.DataFrame([
    {
        "Modelo": d["Modelo"],
        "F1_Enfermo": d["F1_Enfermo"],
        "Macro_F1": d["Macro_F1"],
        "TN": d["TN"],
        "FP": d["FP"],
        "FN": d["FN"],
        "TP": d["TP"]
    }
    for d in diagnosticos_n1
])

display(tabla_n1)

ruta_tabla_n1 = os.path.join(
    RUTA_RESULTADOS_H2,
    "comparacion_nivel1_original_vs_nuevo.csv"
)

tabla_n1.to_csv(
    ruta_tabla_n1,
    index=False,
    encoding="utf-8-sig"
)

print("Guardado en:")
print(ruta_tabla_n1)

## Diagnóstico 2: Rendimiento del Nivel 1 según la procedencia de los textos

El diagnóstico anterior permite comprobar si el nuevo Nivel 1 mejora globalmente, pero no permite saber en qué tipo de publicaciones se producen los errores.

Por este motivo, `test_DU` se divide según la variable `Source`, que identifica el conjunto de datos del que procede cada publicación:

- DATD
- DepressionX
- SDCNL

El objetivo es comprobar si los errores del Nivel 1 se distribuyen de forma similar entre las distintas fuentes o si el cambio a DU binarizado mejora especialmente el comportamiento sobre alguna de ellas.

In [ ]:
# =====================================================
# 41. FUNCIÓN PARA ANALIZAR EL NIVEL 1 POR SOURCE
# =====================================================
#
# ¿Qué queremos saber?
#
# ¿Los errores del Nivel 1 proceden principalmente
# de DATD, DepressionX o SDCNL?
#
# Para cada fuente se calculan:
#
# - total de publicaciones;
# - sanos reales;
# - enfermos reales;
# - TN;
# - FP;
# - FN;
# - TP.
# =====================================================

def diagnostico_nivel1_por_source(
    nombre_modelo,
    predicciones
):

    # Copiar test_DU para conservar Source
    df = test_DU_multiclase.copy()

    # Añadir etiqueta binaria real
    df["Real_binario"] = y_test_DU_binario

    # Añadir predicción realizada por N1
    df["Pred_binario"] = predicciones

    filas = []

    # Analizar cada dataset de procedencia por separado
    for source, grupo in df.groupby("Source"):

        total = len(grupo)

        sanos_reales = (
            grupo["Real_binario"] == 0
        ).sum()

        enfermos_reales = (
            grupo["Real_binario"] == 1
        ).sum()

        # Enfermo real correctamente detectado
        tp = (
            (grupo["Real_binario"] == 1)
            &
            (grupo["Pred_binario"] == 1)
        ).sum()

        # Enfermo real clasificado como sano
        fn = (
            (grupo["Real_binario"] == 1)
            &
            (grupo["Pred_binario"] == 0)
        ).sum()

        # Sano real clasificado como enfermo
        fp = (
            (grupo["Real_binario"] == 0)
            &
            (grupo["Pred_binario"] == 1)
        ).sum()

        # Sano real correctamente detectado
        tn = (
            (grupo["Real_binario"] == 0)
            &
            (grupo["Pred_binario"] == 0)
        ).sum()

        filas.append({
            "Modelo": nombre_modelo,
            "Source": source,
            "Total": int(total),
            "Sanos_reales": int(sanos_reales),
            "Enfermos_reales": int(enfermos_reales),
            "TN": int(tn),
            "FP": int(fp),
            "FN": int(fn),
            "TP": int(tp)
        })

    return pd.DataFrame(filas)

In [ ]:
# =====================================================
# 42. RESUMEN DEL DIAGNÓSTICO 2
# COMPARACIÓN DEL NIVEL 1 POR SOURCE
# =====================================================
#
# Se aplica el mismo análisis a:
#
# - RoBERTa original
# - DeBERTa original
# - RoBERTa nuevo
# - DeBERTa nuevo
#
# De esta forma podemos comprobar cómo cambia el número
# de errores en DATD, DepressionX y SDCNL.
# =====================================================

tabla_source = pd.concat([

    diagnostico_nivel1_por_source(
        "RoBERTa - Nivel 1 original",
        diag_roberta_original["Predicciones"]
    ),

    diagnostico_nivel1_por_source(
        "DeBERTa - Nivel 1 original",
        diag_deberta_original["Predicciones"]
    ),

    diagnostico_nivel1_por_source(
        "RoBERTa - Nuevo Nivel 1",
        diag_roberta_nuevo["Predicciones"]
    ),

    diagnostico_nivel1_por_source(
        "DeBERTa - Nuevo Nivel 1",
        diag_deberta_nuevo["Predicciones"]
    )

], ignore_index=True)

display(tabla_source)

ruta_source = os.path.join(
    RUTA_RESULTADOS_H2,
    "comparacion_nivel1_por_source.csv"
)

tabla_source.to_csv(
    ruta_source,
    index=False,
    encoding="utf-8-sig"
)

print("Guardado en:")
print(ruta_source)

## Diagnóstico 3: Falsos negativos del Nivel 1 en SDCNL

El análisis de la primera hipótesis mostró que el principal problema aparecía en las publicaciones enfermas procedentes de SDCNL.

Estos textos debían ser identificados como enfermos por el Nivel 1 para poder continuar hacia el Nivel 2. Sin embargo, una parte importante era clasificada incorrectamente como sana.

Por ello, este diagnóstico se centra exclusivamente en los textos procedentes de SDCNL y compara cuántos enfermos son detectados correctamente y cuántos se pierden como falsos negativos antes y después de modificar el Nivel 1.

In [ ]:
# =====================================================
# 43. DIAGNÓSTICO 3
# FALSOS NEGATIVOS DEL NIVEL 1 EN SDCNL
# =====================================================
#
# ¿Se ha solucionado el problema detectado en SDCNL?
#
# Se comparan:
#
# - enfermos reales;
# - enfermos detectados correctamente;
# - enfermos clasificados como sanos.
#
# Un enfermo clasificado como sano no llega al Nivel 2.
# =====================================================

tabla_SDCNL = tabla_source[
    tabla_source["Source"]
    .astype(str)
    .str.upper()
    .str.contains("SDCNL")
].copy()

tabla_SDCNL = tabla_SDCNL[
    [
        "Modelo",
        "Enfermos_reales",
        "TP",
        "FN"
    ]
]

tabla_SDCNL = tabla_SDCNL.rename(
    columns={
        "TP": "Enfermos_detectados",
        "FN": "Enfermos_clasificados_como_sanos"
    }
)

display(tabla_SDCNL)

ruta_SDCNL = os.path.join(
    RUTA_RESULTADOS_H2,
    "comparacion_FN_SDCNL_original_vs_nuevo.csv"
)

tabla_SDCNL.to_csv(
    ruta_SDCNL,
    index=False,
    encoding="utf-8-sig"
)

print("Guardado en:")
print(ruta_SDCNL)

In [ ]:
# =====================================================
# 44. FIGURA
# FALSOS NEGATIVOS EN SDCNL
# NIVEL 1 ORIGINAL VS NUEVO NIVEL 1
# =====================================================

fig, ax = plt.subplots(
    figsize=(9, 5)
)

ax.bar(
    tabla_SDCNL["Modelo"],
    tabla_SDCNL[
        "Enfermos_clasificados_como_sanos"
    ]
)

ax.set_ylabel(
    "Falsos negativos"
)

ax.set_title(
    "Enfermos de SDCNL clasificados como sanos por el Nivel 1"
)

ax.tick_params(
    axis="x",
    rotation=25
)

plt.tight_layout()

ruta_figura = os.path.join(
    RUTA_RESULTADOS_H2,
    "comparacion_FN_SDCNL_original_vs_nuevo.png"
)

plt.savefig(
    ruta_figura,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Matrices de confusión de los nuevos modelos del Nivel 1

Para completar el análisis del nuevo Nivel 1 se representan las matrices de confusión de RoBERTa y DeBERTa sobre `test_DU` binarizado.

Estas figuras permiten observar de forma visual los aciertos y errores entre las clases sano y enfermo.

In [ ]:
# =====================================================
# 45. MATRIZ DE CONFUSIÓN
# NUEVO NIVEL 1 - ROBERTA
# =====================================================

cm_roberta_nuevo = confusion_matrix(
    y_test_DU_binario,
    diag_roberta_nuevo["Predicciones"],
    labels=[0, 1]
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_roberta_nuevo,
    display_labels=[
        "Sano",
        "Enfermo"
    ]
)

fig, ax = plt.subplots(
    figsize=(6, 5)
)

disp.plot(
    ax=ax,
    values_format="d"
)

ax.set_title(
    "RoBERTa - Nuevo Nivel 1 - DU binarizado"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        RUTA_RESULTADOS_H2,
        "matriz_confusion_nuevo_N1_RoBERTa.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# =====================================================
# 46. MATRIZ DE CONFUSIÓN
# NUEVO NIVEL 1 - DEBERTA
# =====================================================

cm_deberta_nuevo = confusion_matrix(
    y_test_DU_binario,
    diag_deberta_nuevo["Predicciones"],
    labels=[0, 1]
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_deberta_nuevo,
    display_labels=[
        "Sano",
        "Enfermo"
    ]
)

fig, ax = plt.subplots(
    figsize=(6, 5)
)

disp.plot(
    ax=ax,
    values_format="d"
)

ax.set_title(
    "DeBERTa - Nuevo Nivel 1 - DU binarizado"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        RUTA_RESULTADOS_H2,
        "matriz_confusion_nuevo_N1_DeBERTa.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## Diagnóstico 4: Comparación de la Hipótesis 1 y la Hipótesis 2

Los diagnósticos anteriores permiten comprobar que el nuevo Nivel 1 entrenado con DU binarizado mejora respecto al Nivel 1 de la Hipótesis 1 entrenado con DATD. Sin embargo, es necesario comprobar si esta mejora también se refleja en el funcionamiento completo de la cascada.

Por este motivo, se comparan los resultados finales de:

- los modelos de clasificación directa;
- las cuatro cascadas de la Hipótesis 1, con el Nivel 1 entrenado con DATD;
- las cuatro cascadas de la Hipótesis 2, con el Nivel 1 entrenado con DU binarizado.

Los modelos de clasificación directa no se vuelven a entrenar, ya que no han sufrido ninguna modificación entre ambas hipótesis. Sus resultados se utilizan como referencia para comparar el rendimiento de las dos versiones de la cascada.

Todas las alternativas se evalúan mediante Macro-F1 sobre el mismo `test_DU`.

In [ ]:
# =====================================================
# 47. CARGAR RESULTADOS DE LA HIPÓTESIS 1
# =====================================================
#
# ¿Qué necesitamos recuperar?
#
# Los resultados obtenidos antes de realizar la
# modificación del Nivel 1:
#
# - modelos de clasificación directa;
# - cuatro cascadas de la Hipótesis 1.
#
# Estos resultados ya están guardados en CSV.
#
# NO se vuelve a entrenar ni a ejecutar ningún modelo.
# =====================================================

ruta_resultados_hipotesis1 = os.path.join(
    RUTA_RESULTADOS_ORIGINAL,
    "comparacion_final_directo_vs_cascada.csv"
)

resultados_hipotesis1 = pd.read_csv(
    ruta_resultados_hipotesis1
)

print("Resultados de la Hipótesis 1:")

display(resultados_hipotesis1)

In [ ]:
# =====================================================
# 48. PREPARAR RESULTADOS DE LA HIPÓTESIS 2
# =====================================================
#
# Los resultados de las cuatro cascadas de la
# Hipótesis 2 ya fueron calculados anteriormente.
#
# En esta celda NO se vuelve a ejecutar ninguna cascada.
#
# Únicamente se prepara la tabla para que pueda
# compararse con los resultados de la Hipótesis 1.
# =====================================================

resultados_hipotesis2 = (
    resultados_cascadas_h2.copy()
)

# Cambiar el nombre de la columna para utilizar
# el mismo formato que en los resultados anteriores
resultados_hipotesis2 = (
    resultados_hipotesis2.rename(
        columns={
            "Macro_F1_Test_DU": "F1_Test_DU"
        }
    )
)

# Todas estas filas corresponden a sistemas en cascada
resultados_hipotesis2[
    "Arquitectura"
] = "Cascada"

# Conservar únicamente las columnas necesarias
resultados_hipotesis2 = (
    resultados_hipotesis2[
        [
            "Arquitectura",
            "Nivel_1",
            "Nivel_2",
            "F1_Test_DU"
        ]
    ]
)

print("Resultados de las cascadas de la Hipótesis 2:")

display(resultados_hipotesis2)

In [ ]:
# =====================================================
# 49. IDENTIFICAR LOS RESULTADOS DE CADA HIPÓTESIS
# =====================================================
#
# Para facilitar la comparación se añade una columna
# que indica a qué grupo pertenece cada resultado:
#
# - Modelo directo
# - Hipótesis 1 - Nivel 1 entrenado con DATD
# - Hipótesis 2 - Nivel 1 entrenado con DU binarizado
#
# Los modelos directos se mantienen como referencia,
# ya que son los mismos para ambas hipótesis.
# =====================================================

tabla_hipotesis1 = resultados_hipotesis1.copy()

tabla_hipotesis1["Grupo"] = np.where(
    tabla_hipotesis1["Arquitectura"] == "Directo",
    "Modelo directo",
    "Hipótesis 1 - N1 DATD"
)


tabla_hipotesis2 = resultados_hipotesis2.copy()

tabla_hipotesis2["Grupo"] = (
    "Hipótesis 2 - N1 DU binarizado"
)


print("Resultados de la Hipótesis 1 y modelos directos:")

display(tabla_hipotesis1)


print("\nResultados de la Hipótesis 2:")

display(tabla_hipotesis2)

In [ ]:
# =====================================================
# 50. RESULTADO DEL DIAGNÓSTICO 4
# COMPARACIÓN FINAL ENTRE HIPÓTESIS
# =====================================================
#
# ¿La Hipótesis 2 mejora los resultados de la
# Hipótesis 1?
#
# ¿La mejor cascada de la Hipótesis 2 supera también
# a los modelos de clasificación directa?
#
# Para responder se reúnen:
#
# - modelos directos;
# - cascadas de la Hipótesis 1;
# - cascadas de la Hipótesis 2.
#
# Todos se ordenan según su Macro-F1 sobre test_DU.
# =====================================================

comparacion_hipotesis = pd.concat(
    [
        tabla_hipotesis1,
        tabla_hipotesis2
    ],
    ignore_index=True
)

comparacion_hipotesis = (
    comparacion_hipotesis
    .sort_values(
        "F1_Test_DU",
        ascending=False
    )
    .reset_index(drop=True)
)

display(comparacion_hipotesis)


# -----------------------------------------------------
# GUARDAR COMPARACIÓN
# -----------------------------------------------------

ruta_comparacion_hipotesis = os.path.join(
    RUTA_RESULTADOS_H2,
    "comparacion_hipotesis1_hipotesis2_directo.csv"
)

comparacion_hipotesis.to_csv(
    ruta_comparacion_hipotesis,
    index=False,
    encoding="utf-8-sig"
)

print("Guardado en:")
print(ruta_comparacion_hipotesis)

## Diagnóstico 5: Mejor cascada de la Hipótesis 2

Una vez comparadas las cuatro cascadas de la Hipótesis 2, se selecciona aquella que obtiene el mayor Macro-F1 sobre `test_DU`.

Sobre esta combinación se realiza posteriormente un análisis más detallado para estudiar su comportamiento en las tres clases finales:

- sano;
- depresión;
- suicidio.

Como los resultados de las cuatro cascadas ya fueron calculados anteriormente, no es necesario volver a ejecutarlas todas. Únicamente se volverá a ejecutar la mejor combinación para recuperar sus predicciones individuales sobre `test_DU`.

Esto no implica ningún nuevo entrenamiento: los modelos ya entrenados se cargan desde disco y se utilizan únicamente para realizar predicciones.

In [ ]:
# =====================================================
# 51. DIAGNÓSTICO 5
# SELECCIONAR LA MEJOR CASCADA DE LA HIPÓTESIS 2
# =====================================================
#
# ¿Cuál de las cuatro combinaciones de la Hipótesis 2
# obtiene el mayor Macro-F1 sobre test_DU?
#
# No se ejecuta ningún modelo.
#
# La selección se realiza utilizando los resultados
# que ya fueron guardados anteriormente.
# =====================================================

mejor_cascada_hipotesis2 = (
    resultados_cascadas_h2
    .sort_values(
        "Macro_F1_Test_DU",
        ascending=False
    )
    .iloc[0]
)

print("============================================")
print("MEJOR CASCADA DE LA HIPÓTESIS 2")
print("============================================")

print(
    "Nivel 1:",
    mejor_cascada_hipotesis2["Nivel_1"]
)

print(
    "Nivel 2:",
    mejor_cascada_hipotesis2["Nivel_2"]
)

print(
    "Macro-F1:",
    mejor_cascada_hipotesis2[
        "Macro_F1_Test_DU"
    ]
)

In [ ]:
# =====================================================
# 52. FUNCIÓN PARA EJECUTAR LA MEJOR CASCADA
#     DE LA HIPÓTESIS 2
# =====================================================
#
# ¿Por qué necesitamos volver a ejecutar una cascada?
#
# El CSV anterior contiene el Macro-F1 de cada
# combinación, pero no contiene las predicciones
# individuales de los 1808 textos de test_DU.
#
# Para analizar posteriormente el rendimiento por clase
# necesitamos recuperar esas predicciones.
#
# Por tanto, se vuelve a ejecutar ÚNICAMENTE la mejor
# cascada de la Hipótesis 2.
#
# NO se realiza ningún entrenamiento.
# =====================================================

def predecir_cascada_hipotesis2(
    textos,
    modelo_n1,
    tokenizer_n1,
    modelo_n2,
    tokenizer_n2
):

    # -------------------------------------------------
    # NIVEL 1
    #
    # El nuevo Nivel 1 fue entrenado con DU binarizado:
    #
    # 0 = sano
    # 1 = enfermo
    # -------------------------------------------------

    pred_n1 = predecir_textos(
        modelo_n1,
        tokenizer_n1,
        textos
    )


    # Inicialmente todas las publicaciones se consideran
    # sanas en la predicción final.
    #
    # Las que N1 clasifique como enfermas serán
    # posteriormente modificadas por N2.

    pred_finales = np.zeros(
        len(textos),
        dtype=int
    )


    # Obtener las posiciones de los textos que N1
    # considera enfermos y que, por tanto, deben
    # continuar al Nivel 2.

    indices_enfermos = [
        i
        for i, pred in enumerate(pred_n1)
        if pred == 1
    ]


    print("Total de textos:", len(textos))

    print(
        "Clasificados como sanos en N1:",
        pred_n1.count(0)
    )

    print(
        "Enviados al Nivel 2:",
        len(indices_enfermos)
    )


    # -------------------------------------------------
    # NIVEL 2
    #
    # El Nivel 2 fue entrenado con SDCNL.
    #
    # Internamente:
    #
    # 0 = depresión
    # 1 = suicidio
    #
    # Pero las etiquetas finales de DU son:
    #
    # 1 = depresión
    # 2 = suicidio
    #
    # Por ello se suma 1 a la predicción de N2.
    # -------------------------------------------------

    if len(indices_enfermos) > 0:

        textos_enfermos = [
            textos[i]
            for i in indices_enfermos
        ]

        pred_n2 = predecir_textos(
            modelo_n2,
            tokenizer_n2,
            textos_enfermos
        )

        for indice_original, pred in zip(
            indices_enfermos,
            pred_n2
        ):

            pred_finales[
                indice_original
            ] = pred + 1


    return pred_finales

In [ ]:
# =====================================================
# 53. RECUPERAR LAS PREDICCIONES DE LA MEJOR CASCADA
#     DE LA HIPÓTESIS 2
# =====================================================
#
# Se identifica automáticamente qué modelo utiliza
# la combinación ganadora en cada nivel.
#
# Después se cargan ambos modelos desde disco y se
# realizan predicciones sobre test_DU.
#
# NO se realiza ningún entrenamiento.
# =====================================================


# -----------------------------------------------------
# IDENTIFICAR LOS MODELOS GANADORES
# -----------------------------------------------------

nombre_n1_mejor = (
    mejor_cascada_hipotesis2["Nivel_1"]
)

nombre_n2_mejor = (
    mejor_cascada_hipotesis2["Nivel_2"]
)


# -----------------------------------------------------
# ELEGIR EL MODELO DEL NIVEL 1
# -----------------------------------------------------

if nombre_n1_mejor == "RoBERTa":

    ruta_n1_mejor = (
        RUTA_N1_ROBERTA_NUEVO
    )

else:

    ruta_n1_mejor = (
        RUTA_N1_DEBERTA_NUEVO
    )


# -----------------------------------------------------
# ELEGIR EL MODELO DEL NIVEL 2
# -----------------------------------------------------

if nombre_n2_mejor == "RoBERTa":

    ruta_n2_mejor = (
        RUTA_N2_ROBERTA
    )

else:

    ruta_n2_mejor = (
        RUTA_N2_DEBERTA
    )


print(
    "Cargando mejor cascada de la Hipótesis 2:",
    nombre_n1_mejor,
    "->",
    nombre_n2_mejor
)


# -----------------------------------------------------
# CARGAR LOS DOS MODELOS
# -----------------------------------------------------

modelo_n1_mejor, tokenizer_n1_mejor = (
    cargar_modelo_entrenado(
        ruta_n1_mejor
    )
)

modelo_n2_mejor, tokenizer_n2_mejor = (
    cargar_modelo_entrenado(
        ruta_n2_mejor
    )
)


# -----------------------------------------------------
# REALIZAR LAS PREDICCIONES
# -----------------------------------------------------

pred_mejor_cascada_hipotesis2 = (
    predecir_cascada_hipotesis2(
        textos_test_DU,
        modelo_n1_mejor,
        tokenizer_n1_mejor,
        modelo_n2_mejor,
        tokenizer_n2_mejor
    )
)


# -----------------------------------------------------
# COMPROBAR QUE EL MACRO-F1 COINCIDE CON EL QUE
# HABÍAMOS GUARDADO ANTERIORMENTE
# -----------------------------------------------------

macro_f1_comprobacion = f1_score(
    y_test_DU,
    pred_mejor_cascada_hipotesis2,
    average="macro"
)

print(
    "\nMacro-F1 obtenido ahora:",
    macro_f1_comprobacion
)

print(
    "Macro-F1 guardado anteriormente:",
    mejor_cascada_hipotesis2[
        "Macro_F1_Test_DU"
    ]
)


# -----------------------------------------------------
# LIBERAR MEMORIA
# -----------------------------------------------------

del modelo_n1_mejor
del tokenizer_n1_mejor
del modelo_n2_mejor
del tokenizer_n2_mejor

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

### Rendimiento por clase de la mejor cascada de la Hipótesis 2

El Macro-F1 permite resumir el rendimiento global del modelo teniendo en cuenta las tres clases, pero no permite observar cómo se comporta de forma individual en sano, depresión y suicidio.

Por este motivo, se calculan las métricas de precisión, exhaustividad y F1 para cada una de las tres clases de `test_DU`.

In [ ]:
# =====================================================
# 54. MÉTRICAS POR CLASE
# MEJOR CASCADA DE LA HIPÓTESIS 2
# =====================================================
#
# ¿La mejor cascada de la Hipótesis 2 funciona igual
# de bien para las tres clases?
#
# Se analiza por separado:
#
# - Sano
# - Depresión
# - Suicidio
#
# Para cada clase se obtienen:
#
# - Precisión
# - Exhaustividad
# - F1
# - Número de ejemplos
# =====================================================

reporte_mejor_cascada_hipotesis2 = (
    classification_report(

        y_test_DU,

        pred_mejor_cascada_hipotesis2,

        labels=[0, 1, 2],

        target_names=[
            "Sano",
            "Depresión",
            "Suicidio"
        ],

        output_dict=True,

        zero_division=0
    )
)


metricas_mejor_cascada_hipotesis2 = (
    pd.DataFrame(
        reporte_mejor_cascada_hipotesis2
    ).T
)


display(
    metricas_mejor_cascada_hipotesis2
)


# -----------------------------------------------------
# GUARDAR RESULTADOS
# -----------------------------------------------------

ruta_metricas = os.path.join(
    RUTA_RESULTADOS_H2,
    "metricas_por_clase_mejor_cascada_hipotesis2.csv"
)

metricas_mejor_cascada_hipotesis2.to_csv(
    ruta_metricas,
    encoding="utf-8-sig"
)

print("Guardado en:")
print(ruta_metricas)

In [ ]:
# =====================================================
# 55. MATRIZ DE CONFUSIÓN
# MEJOR CASCADA DE LA HIPÓTESIS 2
# =====================================================
#
# ¿Entre qué clases se producen los errores de la
# mejor cascada de la Hipótesis 2?
#
# La matriz permite observar las confusiones entre:
#
# - sano;
# - depresión;
# - suicidio.
# =====================================================

cm_mejor_cascada_hipotesis2 = confusion_matrix(
    y_test_DU,
    pred_mejor_cascada_hipotesis2,
    labels=[0, 1, 2]
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_mejor_cascada_hipotesis2,
    display_labels=[
        "Sano",
        "Depresión",
        "Suicidio"
    ]
)

fig, ax = plt.subplots(
    figsize=(7, 6)
)

disp.plot(
    ax=ax,
    values_format="d"
)

ax.set_title(
    "Mejor cascada de la Hipótesis 2"
)

plt.tight_layout()


# -----------------------------------------------------
# GUARDAR FIGURA
# -----------------------------------------------------

ruta_matriz = os.path.join(
    RUTA_RESULTADOS_H2,
    "matriz_confusion_mejor_cascada_hipotesis2.png"
)

plt.savefig(
    ruta_matriz,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Guardado en:")
print(ruta_matriz)

## Diagnóstico final: Clasificación directa frente a Hipótesis 1 e Hipótesis 2

Finalmente, se comparan los mejores resultados obtenidos mediante las tres alternativas principales del trabajo:

- el mejor modelo de clasificación directa;
- la mejor cascada de la Hipótesis 1;
- la mejor cascada de la Hipótesis 2.

Esta comparación permite determinar si la modificación introducida en la Hipótesis 2 consigue corregir el problema detectado en la primera cascada y si el nuevo sistema supera también a la clasificación directa.

Además, se calcula la mejora de la Hipótesis 2 respecto a la Hipótesis 1 y su diferencia respecto al mejor modelo directo.

Las métricas del Nivel 1 analizadas en los diagnósticos anteriores no se comparan directamente con estas métricas finales, ya que corresponden a tareas diferentes:

- Nivel 1: sano frente a enfermo.
- Clasificación final: sano frente a depresión frente a suicidio.

In [ ]:
# =====================================================
# 56. RESUMEN FINAL DEL BLOQUE DE DIAGNÓSTICO
# =====================================================
#
# ¿Cuál es el mejor modelo de clasificación directa?
#
# ¿Cuál es la mejor combinación de modelos de la
# Hipótesis 1?
#
# ¿Cuál es la mejor combinación de modelos de la
# Hipótesis 2?
#
# En lugar de mostrar únicamente el Macro-F1, se indica
# también qué modelos forman cada sistema para que el
# resultado pueda interpretarse directamente.
#
# Finalmente se calcula cuánto mejora la mejor cascada
# de la Hipótesis 2 respecto a:
#
# - la mejor cascada de la Hipótesis 1;
# - el mejor modelo de clasificación directa.
# =====================================================


# -----------------------------------------------------
# 1. MEJOR MODELO DE CLASIFICACIÓN DIRECTA
# -----------------------------------------------------

fila_mejor_directo = (
    resultados_hipotesis1[
        resultados_hipotesis1["Arquitectura"] == "Directo"
    ]
    .sort_values(
        "F1_Test_DU",
        ascending=False
    )
    .iloc[0]
)

mejor_directo = fila_mejor_directo["F1_Test_DU"]

# En los modelos directos el modelo utilizado aparece
# en la columna Nivel_1
modelo_mejor_directo = fila_mejor_directo["Nivel_1"]


# -----------------------------------------------------
# 2. MEJOR CASCADA DE LA HIPÓTESIS 1
#
# Nivel 1 entrenado con DATD
# Nivel 2 entrenado con SDCNL
# -----------------------------------------------------

fila_mejor_cascada_hipotesis1 = (
    resultados_hipotesis1[
        resultados_hipotesis1["Arquitectura"] == "Cascada"
    ]
    .sort_values(
        "F1_Test_DU",
        ascending=False
    )
    .iloc[0]
)

mejor_cascada_hipotesis1 = (
    fila_mejor_cascada_hipotesis1["F1_Test_DU"]
)

modelo_n1_hipotesis1 = (
    fila_mejor_cascada_hipotesis1["Nivel_1"]
)

modelo_n2_hipotesis1 = (
    fila_mejor_cascada_hipotesis1["Nivel_2"]
)


# -----------------------------------------------------
# 3. MEJOR CASCADA DE LA HIPÓTESIS 2
#
# Nivel 1 entrenado con DU binarizado
# Nivel 2 entrenado con SDCNL
# -----------------------------------------------------

fila_mejor_cascada_hipotesis2 = (
    resultados_cascadas_h2
    .sort_values(
        "Macro_F1_Test_DU",
        ascending=False
    )
    .iloc[0]
)

mejor_cascada_hipotesis2_f1 = (
    fila_mejor_cascada_hipotesis2[
        "Macro_F1_Test_DU"
    ]
)

modelo_n1_hipotesis2 = (
    fila_mejor_cascada_hipotesis2["Nivel_1"]
)

modelo_n2_hipotesis2 = (
    fila_mejor_cascada_hipotesis2["Nivel_2"]
)


# -----------------------------------------------------
# 4. CALCULAR LAS MEJORAS
# -----------------------------------------------------

mejora_hipotesis2_vs_hipotesis1 = (
    mejor_cascada_hipotesis2_f1
    -
    mejor_cascada_hipotesis1
)

mejora_hipotesis2_vs_directo = (
    mejor_cascada_hipotesis2_f1
    -
    mejor_directo
)


# -----------------------------------------------------
# 5. CREAR TABLA RESUMEN
# -----------------------------------------------------

resumen_final = pd.DataFrame({

    "Sistema": [

        "Clasificación directa",

        "Cascada Hipótesis 1",

        "Cascada Hipótesis 2"
    ],

    "Nivel_1": [

        modelo_mejor_directo,

        modelo_n1_hipotesis1,

        modelo_n1_hipotesis2
    ],

    "Entrenamiento_Nivel_1": [

        "DU multiclase",

        "DATD",

        "DU binarizado"
    ],

    "Nivel_2": [

        "No aplica",

        modelo_n2_hipotesis1,

        modelo_n2_hipotesis2
    ],

    "Entrenamiento_Nivel_2": [

        "No aplica",

        "SDCNL",

        "SDCNL"
    ],

    "Macro_F1_test_DU": [

        mejor_directo,

        mejor_cascada_hipotesis1,

        mejor_cascada_hipotesis2_f1
    ]
})


display(resumen_final)


# -----------------------------------------------------
# 6. MOSTRAR LAS MEJORAS POR SEPARADO
# -----------------------------------------------------

comparacion_mejoras = pd.DataFrame({

    "Comparación": [

        "Hipótesis 2 respecto a Hipótesis 1",

        "Hipótesis 2 respecto a clasificación directa"
    ],

    "Diferencia_Macro_F1": [

        mejora_hipotesis2_vs_hipotesis1,

        mejora_hipotesis2_vs_directo
    ]
})


display(comparacion_mejoras)


# -----------------------------------------------------
# 7. GUARDAR LOS DOS RESÚMENES
# -----------------------------------------------------

ruta_resumen = os.path.join(
    RUTA_RESULTADOS_H2,
    "resumen_final_hipotesis2.csv"
)

resumen_final.to_csv(
    ruta_resumen,
    index=False,
    encoding="utf-8-sig"
)


ruta_mejoras = os.path.join(
    RUTA_RESULTADOS_H2,
    "mejoras_hipotesis2.csv"
)

comparacion_mejoras.to_csv(
    ruta_mejoras,
    index=False,
    encoding="utf-8-sig"
)


print("Resumen final guardado correctamente.")

In [ ]:
# =====================================================
# 57. CONCLUSIÓN FINAL DE LA SEGUNDA HIPÓTESIS
# =====================================================
#
# Esta celda muestra de forma explícita:
#
# 1. Qué modelo directo obtiene el mejor resultado.
# 2. Qué combinación gana en la Hipótesis 1.
# 3. Qué combinación gana en la Hipótesis 2.
# 4. Qué cambia entre ambas hipótesis.
# 5. Cuánto mejora el resultado final.
# =====================================================


print("==========================================================")
print("CONCLUSIÓN FINAL DE LA SEGUNDA HIPÓTESIS")
print("==========================================================")


# -----------------------------------------------------
# CLASIFICACIÓN DIRECTA
# -----------------------------------------------------

print("\n1. MEJOR MODELO DE CLASIFICACIÓN DIRECTA")
print("------------------------------------------")

print(
    f"Modelo: {modelo_mejor_directo}"
)

print(
    "Entrenamiento: DU multiclase "
    "(sano / depresión / suicidio)"
)

print(
    f"Macro-F1 sobre test_DU: "
    f"{mejor_directo:.4f}"
)


# -----------------------------------------------------
# HIPÓTESIS 1
# -----------------------------------------------------

print("\n2. MEJOR CASCADA DE LA HIPÓTESIS 1")
print("-----------------------------------")

print(
    f"Nivel 1: {modelo_n1_hipotesis1} "
    f"entrenado con DATD"
)

print(
    f"Nivel 2: {modelo_n2_hipotesis1} "
    f"entrenado con SDCNL"
)

print(
    f"Combinación: "
    f"{modelo_n1_hipotesis1} -> "
    f"{modelo_n2_hipotesis1}"
)

print(
    f"Macro-F1 sobre test_DU: "
    f"{mejor_cascada_hipotesis1:.4f}"
)


# -----------------------------------------------------
# HIPÓTESIS 2
# -----------------------------------------------------

print("\n3. MEJOR CASCADA DE LA HIPÓTESIS 2")
print("-----------------------------------")

print(
    f"Nivel 1: {modelo_n1_hipotesis2} "
    f"entrenado con DU binarizado"
)

print(
    f"Nivel 2: {modelo_n2_hipotesis2} "
    f"entrenado con SDCNL"
)

print(
    f"Combinación: "
    f"{modelo_n1_hipotesis2} -> "
    f"{modelo_n2_hipotesis2}"
)

print(
    f"Macro-F1 sobre test_DU: "
    f"{mejor_cascada_hipotesis2_f1:.4f}"
)


# -----------------------------------------------------
# COMPARACIÓN ENTRE HIPÓTESIS
# -----------------------------------------------------

print("\n4. COMPARACIÓN ENTRE HIPÓTESIS")
print("-------------------------------")

print(
    "Hipótesis 1:"
)

print(
    f"  {modelo_n1_hipotesis1} entrenado con DATD"
    f" -> {modelo_n2_hipotesis1} entrenado con SDCNL"
)

print(
    f"  Macro-F1 = "
    f"{mejor_cascada_hipotesis1:.4f}"
)


print(
    "\nHipótesis 2:"
)

print(
    f"  {modelo_n1_hipotesis2} entrenado con DU binarizado"
    f" -> {modelo_n2_hipotesis2} entrenado con SDCNL"
)

print(
    f"  Macro-F1 = "
    f"{mejor_cascada_hipotesis2_f1:.4f}"
)


print(
    "\nMejora de la Hipótesis 2 respecto "
    "a la Hipótesis 1:"
)

print(
    f"  +{mejora_hipotesis2_vs_hipotesis1:.4f} "
    "Macro-F1"
)


# -----------------------------------------------------
# COMPARACIÓN CON EL MODELO DIRECTO
# -----------------------------------------------------

print("\n5. COMPARACIÓN CON LA CLASIFICACIÓN DIRECTA")
print("--------------------------------------------")

print(
    f"Mejor modelo directo ({modelo_mejor_directo}): "
    f"{mejor_directo:.4f}"
)

print(
    f"Mejor cascada Hipótesis 2 "
    f"({modelo_n1_hipotesis2} -> "
    f"{modelo_n2_hipotesis2}): "
    f"{mejor_cascada_hipotesis2_f1:.4f}"
)

print(
    "\nDiferencia a favor de la Hipótesis 2:"
)

print(
    f"  +{mejora_hipotesis2_vs_directo:.4f} "
    "Macro-F1"
)


# -----------------------------------------------------
# CONCLUSIÓN
# -----------------------------------------------------

print("\n==========================================================")
print("RESULTADO")
print("==========================================================")

print(
    "La Hipótesis 2 obtiene el mejor resultado final."
)

print(
    f"La mejor configuración es "
    f"{modelo_n1_hipotesis2} en el Nivel 1, "
    f"entrenado con DU binarizado, seguido de "
    f"{modelo_n2_hipotesis2} en el Nivel 2, "
    f"entrenado con SDCNL."
)

print(
    f"Esta combinación alcanza un Macro-F1 de "
    f"{mejor_cascada_hipotesis2_f1:.4f} sobre test_DU."
)